# Mean-Variance Portfolio Optimization

From-scratch implementation of Markowitz mean-variance optimization, efficient frontier computation, and the capital allocation line.

**Outline**
1. Why not just pick the highest return?
2. Portfolio return: the easy part
3. Portfolio risk: the surprising part (correlation!)
4. The 2-asset case worked through with numbers
5. The N-asset generalization
6. The efficient frontier: the best you can do
7. Minimum variance portfolio
8. Efficient frontier via Lagrange multipliers
9. Two-fund separation theorem
10. Short-sale constraints
11. Adding a risk-free asset: Capital Allocation Line
12. The tangency portfolio and the Sharpe ratio
13. Putting it all together
14. Summary of key formulas
15. References
> **Prerequisites:** You should be comfortable with expected return, standard deviation, and the concept of correlation between asset returns. Knowledge of matrix algebra is helpful but not required — we explain the key operations as we go.


---
## 1. Why Not Just Pick the Highest Return?

Imagine you are choosing between two investments:

| Investment | Expected Return | Risk (Std Dev) |
|:--|:--|:--|
| Stock A | 15% per year | 30% |
| Stock B | 10% per year | 12% |

If you only cared about return, you would always pick Stock A. But Stock A's return could easily swing from +45% to -15% in a single year (roughly two standard deviations). Stock B is far more predictable.

**The fundamental insight of modern portfolio theory is that investors care about both return and risk.** A rational investor will only take on more risk if they are compensated with higher expected return.

Harry Markowitz formalized this in his 1952 paper "Portfolio Selection." His insight was deceptively simple: don't just look at individual assets -- look at how they behave *together*. When you combine assets into a portfolio, something remarkable happens: **you can often reduce risk without sacrificing return** (or increase return without adding risk).

This is the foundation of everything in this notebook.

> **Key Concept:** Mean-variance analysis is the framework for making optimal investment decisions when you care about both how much you expect to earn (the *mean* return) and how uncertain that return is (the *variance* or standard deviation).

> **CFA Exam Tip:** The CFA curriculum emphasizes that Markowitz portfolio theory assumes investors are *risk-averse* -- given two portfolios with the same expected return, they always prefer the one with lower risk. This is the behavioral foundation of the entire framework.

### A Real-World Analogy

Think of it like choosing a route to work. One route takes 20 minutes on average but can range from 10 to 50 minutes depending on traffic. Another route takes 25 minutes on average but never varies by more than 5 minutes. If you have a meeting to catch, you might prefer the slower but more reliable route. That is risk aversion in action -- you are willing to give up some expected speed (return) in exchange for less uncertainty (risk).### The Risk-Return Trade-off

This observation — that risk matters, not just return — is the foundation of **Modern Portfolio Theory (MPT)**, developed by Harry Markowitz in 1952. Before Markowitz, most investors evaluated assets individually. Markowitz showed that the risk of a portfolio depends not only on individual asset risks but on how assets **co-move** (their correlations).

> **Key Concept:** Markowitz's fundamental insight: **you should evaluate investments in the context of a portfolio, not in isolation.** An asset that looks risky by itself might actually REDUCE portfolio risk if it has low correlation with your other holdings.

This single idea revolutionised finance and earned Markowitz the Nobel Prize in Economics (1990).


---
## 2. Portfolio Return: The Easy Part

The expected return of a portfolio is simply the **weighted average** of the individual asset returns. There is nothing surprising here -- it is completely intuitive.

### In Plain English

If you put 60% of your money in stocks (expected return 10%) and 40% in bonds (expected return 4%), your portfolio's expected return is:

$$0.60 \times 10\% + 0.40 \times 4\% = 6.0\% + 1.6\% = 7.6\%$$

Nothing fancy. Just a weighted average.

### The Formula

**For two assets:**

$$E[R_P] = w_1 E[R_1] + w_2 E[R_2]$$

where:
- $w_1, w_2$ = the fraction of your wealth in each asset (weights must sum to 1)
- $E[R_1], E[R_2]$ = expected returns of the individual assets

**For N assets (vector notation):**

$$E[R_P] = \mathbf{w}^\top \boldsymbol{\mu} = \sum_{i=1}^{N} w_i \mu_i$$

where $\mathbf{w}$ is the N-by-1 weight vector and $\boldsymbol{\mu}$ is the N-by-1 expected return vector.

### Worked Example

| Asset | Weight | Expected Return | Contribution |
|:--|:--|:--|:--|
| US Equity | 50% | 10% | 5.00% |
| Int'l Equity | 20% | 12% | 2.40% |
| Bonds | 30% | 4% | 1.20% |
| **Portfolio** | **100%** | | **8.60%** |

Simple arithmetic. The portfolio expected return is just 5.00% + 2.40% + 1.20% = 8.60%.

> **Key Concept:** Portfolio return is always a weighted average -- no surprises. The magic happens with risk.

> **CFA Exam Tip:** On the exam, you may see a question asking for the expected return of a 3-asset portfolio. Just multiply each weight by each return and sum. This is the simplest calculation in portfolio theory -- do not overthink it.

---
## 3. Portfolio Risk: The Surprising Part

Here is where Markowitz's insight becomes powerful. **Portfolio risk is NOT the weighted average of individual risks.** It can be much lower, thanks to diversification.

### Why? Because assets don't move in lockstep.

Think of it this way: on a day when your stocks go down 2%, your bonds might go *up* 1%. The losses and gains partially cancel out, making the portfolio smoother than either asset alone. This cancellation effect depends on the **correlation** between the assets.

**Correlation ($\rho$)** ranges from -1 to +1:
- **$\rho = +1$**: Assets always move in the same direction by proportional amounts. No diversification benefit.
- **$\rho = 0$**: Assets move independently. Significant diversification benefit.
- **$\rho = -1$**: Assets always move in opposite directions. Maximum diversification -- you can theoretically eliminate all risk.

### The 2-Asset Formula

$$\sigma_P^2 = w_1^2 \sigma_1^2 + w_2^2 \sigma_2^2 + 2 w_1 w_2 \rho_{12} \sigma_1 \sigma_2$$

where:
- $\sigma_P^2$ = portfolio variance
- $\sigma_1, \sigma_2$ = standard deviations (volatilities) of each asset
- $\rho_{12}$ = correlation between the two assets
- $2 w_1 w_2 \rho_{12} \sigma_1 \sigma_2$ = the **cross-term** (the covariance piece)

**The third term is the key.** It is the only place correlation appears. When $\rho_{12} < 1$, this term is smaller than it would be if the assets moved in perfect lockstep, and the portfolio variance is **less than** the weighted average variance.

### Worked Example with Actual Numbers

**Setup:** Two assets with equal 50/50 weighting.
- Asset 1: $\sigma_1 = 12\%$, Asset 2: $\sigma_2 = 25\%$
- If we simply averaged the risks: $0.5 \times 12\% + 0.5 \times 25\% = 18.5\%$

Now let's compute actual portfolio risk for different correlations:

| Correlation ($\rho$) | Portfolio Std Dev | vs. Naive Average (18.5%) |
|:--|:--|:--|
| +1.0 (perfect) | 18.50% | No benefit |
| +0.5 | 16.36% | 2.14% reduction |
| 0.0 (uncorrelated) | 13.87% | 4.63% reduction |
| -0.5 | 10.81% | 7.69% reduction |
| -1.0 (perfect neg.) | 6.50% | 12.00% reduction |

Let's verify the $\rho = 0$ row step by step:

$$\sigma_P^2 = (0.5)^2(0.12)^2 + (0.5)^2(0.25)^2 + 2(0.5)(0.5)(0)(0.12)(0.25)$$
$$= 0.25 \times 0.0144 + 0.25 \times 0.0625 + 0$$
$$= 0.0036 + 0.015625 = 0.019225$$
$$\sigma_P = \sqrt{0.019225} = 13.87\%$$

That is a huge reduction -- from 18.5% to 13.87% -- and we did not sacrifice any expected return!

> **Key Concept:** Diversification works because of imperfect correlation. The lower the correlation between assets, the greater the risk reduction. This is the "free lunch" of finance -- you can reduce risk without reducing expected return.

> **CFA Exam Tip:** You MUST be able to calculate portfolio variance for a 2-asset portfolio by hand. The formula is tested frequently. Remember: the cross-term (covariance) is what makes portfolio risk different from the weighted average of individual risks. Many exam questions hinge on recognizing that $\sigma_{12} = \rho_{12} \sigma_1 \sigma_2$.

### Special Cases Worth Memorizing

- **$\rho = +1$:** No diversification benefit. Portfolio risk IS the weighted average: $\sigma_P = w_1 \sigma_1 + w_2 \sigma_2$.
- **$\rho = -1$:** Maximum diversification. You can create a **zero-risk portfolio** with weights $w_1 = \frac{\sigma_2}{\sigma_1 + \sigma_2}$ and $w_2 = \frac{\sigma_1}{\sigma_1 + \sigma_2}$.
- **$\rho = 0$:** Substantial diversification benefit, especially when both assets have similar volatility.

---
## 4. Visualizing the 2-Asset Case

Let's see this in action. We plot all possible combinations of our two assets for different correlation values. Each curve traces the risk-return combinations available as we vary the weight on Asset 1.

**What to look for in the plot:**
- When correlation is +1, the curve is a straight line (no diversification benefit).
- As correlation decreases, the curves bow further to the left -- the same return is achievable at lower risk.
- The leftward "bulge" IS the diversification benefit, visualized.

We use Asset 1 with $\mu_1 = 10\%, \sigma_1 = 12\%$ and Asset 2 with $\mu_2 = 18\%, \sigma_2 = 25\%$.### What to Watch For

As we sweep the portfolio weight from 0% to 100% in Asset A, the portfolio traces a curve in risk-return space. The shape of this curve depends entirely on the **correlation** $\rho$ between the two assets:
- $\rho = 1$: straight line (no diversification benefit)
- $\rho = 0$: a curve that bows to the left (moderate benefit)
- $\rho = -1$: two straight lines meeting at $\sigma = 0$ (perfect hedge possible)

Let's see this:


In [ ]:
%matplotlib inline
import numpy as np
from scipy import stats, optimize, linalg
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
rng = np.random.default_rng(SEED)

ATOL = 1e-10
RTOL = 1e-6

PRIMARY   = 'steelblue'
SECONDARY = 'coral'
TERTIARY  = 'seagreen'
ACCENT    = 'gold'
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})

In [ ]:
# --- Two-Asset Case ---
# Asset parameters (annualized)
mu1, mu2 = 0.10, 0.18       # expected returns: 10% and 18%
sig1, sig2 = 0.12, 0.25     # volatilities: 12% and 25%

# Vary the weight on Asset 1 from -50% to 150% (allowing short sales for now)
w1 = np.linspace(-0.5, 1.5, 300)
w2 = 1 - w1

fig, ax = plt.subplots()
for rho, c, ls in [(-0.5, TERTIARY, '-'), (0.0, PRIMARY, '-'), (0.5, SECONDARY, '-'), (1.0, 'grey', '--')]:
    # Portfolio return is the same regardless of correlation
    port_ret = w1 * mu1 + w2 * mu2
    # Portfolio variance depends on correlation -- this is the key insight
    port_var = w1**2 * sig1**2 + w2**2 * sig2**2 + 2 * w1 * w2 * rho * sig1 * sig2
    port_std = np.sqrt(np.maximum(port_var, 0))  # avoid numerical negatives
    ax.plot(port_std * 100, port_ret * 100, color=c, linewidth=2, linestyle=ls, label=f'\u03c1 = {rho}')

# Mark the two individual assets
ax.plot(sig1 * 100, mu1 * 100, 'o', color='black', markersize=10, zorder=5)
ax.plot(sig2 * 100, mu2 * 100, 's', color='black', markersize=10, zorder=5)
ax.annotate('Asset 1', (sig1 * 100 + 0.5, mu1 * 100))
ax.annotate('Asset 2', (sig2 * 100 + 0.5, mu2 * 100))
ax.set_xlabel('Portfolio Std Dev (%)')
ax.set_ylabel('Portfolio Expected Return (%)')
ax.set_title('2-Asset Portfolios \u2014 Effect of Correlation')
ax.legend()
plt.tight_layout()
plt.show()

### Interpreting the Plot

Notice several things:

1. **The grey dashed line ($\rho = 1$)** is straight -- when assets are perfectly correlated, combining them gives no diversification benefit. The portfolio risk is just the weighted average.

2. **As correlation decreases,** the curves bow further to the left. This leftward bulge IS the diversification benefit -- you are getting the same return at lower risk.

3. **At $\rho = -0.5$,** the curve reaches very close to the y-axis, meaning you can construct a portfolio with very low risk that still earns a reasonable return.

4. **The top half of each curve is "efficient"** (you get more return for more risk), while the bottom half is "dominated" (you could get more return at the same risk level by moving to the upper part).

> **Key Concept:** The leftward "bulge" of the portfolio opportunity set is the visual signature of the diversification benefit. The lower the correlation, the bigger the bulge, and the more risk you can eliminate.

> **CFA Exam Tip:** On the exam, you may be asked to identify which correlation produces the greatest diversification benefit. The answer is always the lowest correlation (most negative). At $\rho = -1$, you can theoretically eliminate all risk.> **Key Concept:** The key takeaway is that **correlation determines the shape of the opportunity set**. Lower correlation → more "bowing" to the left → more diversification benefit. At $\rho = -1$, you can construct a zero-risk portfolio. At $\rho = +1$, diversification provides no risk reduction at all.

> **CFA Exam Tip:** The exam loves asking what happens to the efficient frontier as correlation changes. Remember: lower $\rho$ → frontier bows more to the left → greater diversification benefit. The minimum variance portfolio has lower risk with lower $\rho$.


---
## 5. The N-Asset Generalization

Real portfolios contain many assets. The 2-asset formulas extend naturally using matrix notation.

### Portfolio Return (same idea, just more assets)

$$E[R_P] = \mathbf{w}^\top \boldsymbol{\mu} = \sum_{i=1}^{N} w_i \mu_i$$

Still just a weighted average. Nothing new.

### Portfolio Variance (where the magic is)

$$\sigma_P^2 = \mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w} = \sum_{i=1}^{N} \sum_{j=1}^{N} w_i w_j \sigma_{ij}$$

where $\boldsymbol{\Sigma}$ is the **covariance matrix** -- an $N \times N$ matrix where:
- Diagonal entries are the variances: $\sigma_{ii} = \sigma_i^2$
- Off-diagonal entries are the covariances: $\sigma_{ij} = \rho_{ij} \sigma_i \sigma_j$

**Building the covariance matrix from correlations and volatilities:**

$$\boldsymbol{\Sigma} = \text{diag}(\boldsymbol{\sigma}) \times \mathbf{C} \times \text{diag}(\boldsymbol{\sigma})$$

### How Many Parameters?

For $N$ assets, the covariance matrix has $\frac{N(N+1)}{2}$ unique entries (it is symmetric). For 5 assets that is 15. For 500 stocks, that is 125,250 parameters -- estimation is challenging in practice.

> **CFA Exam Tip:** You need to know that the covariance matrix is symmetric ($\sigma_{ij} = \sigma_{ji}$) and positive semi-definite. The number of unique parameters is $N(N+1)/2$.### Estimation Challenges

For $N$ assets, we need to estimate:
- $N$ expected returns
- $N$ variances
- $N(N-1)/2$ pairwise covariances

| Assets ($N$) | Parameters to estimate |
|:---:|:---:|
| 5 | 20 |
| 50 | 1,325 |
| 500 | 125,750 |

> **Common Mistake:** With finite data, estimated covariance matrices are noisy. Small estimation errors can lead to wildly different "optimal" portfolios. This is why Markowitz optimisation is sometimes called an "error maximiser" — it overweights assets with overestimated returns and underweights those with underestimated returns. Practical solutions include shrinkage estimators (Ledoit-Wolf) and factor models.


In [ ]:
# --- 5-Asset Universe ---
# These parameters are stylized but realistic for major asset classes
N = 5
asset_names = ['Equity US', 'Equity Intl', 'Bonds', 'Real Estate', 'Commodities']

# Expected annual returns
mu = np.array([0.10, 0.12, 0.04, 0.08, 0.07])

# Annual volatilities
sigma = np.array([0.16, 0.22, 0.05, 0.14, 0.20])

# Correlation matrix -- note the low/negative correlations between bonds and equities
corr = np.array([
    [1.00, 0.75, 0.10, 0.50, 0.20],   # US Equity
    [0.75, 1.00, 0.05, 0.45, 0.30],   # Intl Equity
    [0.10, 0.05, 1.00, 0.15, -0.10],  # Bonds (low correlation with everything!)
    [0.50, 0.45, 0.15, 1.00, 0.25],   # Real Estate
    [0.20, 0.30, -0.10, 0.25, 1.00],  # Commodities
])

# Construct the covariance matrix: Sigma = diag(sigma) @ Corr @ diag(sigma)
Sigma = np.diag(sigma) @ corr @ np.diag(sigma)
assert np.allclose(Sigma, Sigma.T)  # covariance matrix must be symmetric

print("Expected returns:", mu)
print("Volatilities:    ", sigma)
print("\nCovariance matrix:")
print(np.round(Sigma, 5))

### Reading the Covariance Matrix

A few things to notice in the output above:

- **Bonds** have very low covariance with equities -- this is why bonds are the classic diversifier. The correlation of bonds with US Equity is only 0.10, and with International Equity only 0.05.
- **US and International Equity** have high covariance (0.75 correlation) -- they tend to move together, so holding both gives limited diversification benefit.
- **Commodities** have near-zero or slightly negative correlation with bonds (-0.10) -- another useful diversification pair.

> **Key Concept:** The covariance matrix is the most important input to portfolio optimization. It encodes ALL the diversification relationships between assets. Garbage in, garbage out.> **Key Concept:** The covariance matrix $\Sigma$ contains ALL the information about how assets co-move. The diagonal entries are variances (individual risk); the off-diagonal entries are covariances (co-movement). A good portfolio exploits low or negative off-diagonal entries to reduce total portfolio risk.

> **Common Mistake:** Students sometimes confuse the covariance matrix with the correlation matrix. Covariance depends on the scale of returns; correlation is standardised between $-1$ and $+1$. The relationship is $\sigma_{ij} = \rho_{ij} \sigma_i \sigma_j$.


---
## 6. The Efficient Frontier: The Best You Can Do

For every possible target return, there exists one portfolio with the *minimum possible risk*. The collection of all such portfolios is the **efficient frontier**.

### Think of It as a Menu

The efficient frontier shows you, for each level of risk you are willing to accept, the absolute maximum return you can achieve. If your portfolio lies below the frontier, you are leaving money on the table.

### The Optimization Problem

$$\min_{\mathbf{w}} \quad \mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w}$$
$$\text{subject to} \quad \mathbf{w}^\top \boldsymbol{\mu} = \mu_{\text{target}}, \quad \mathbf{w}^\top \mathbf{1} = 1$$

- First constraint: "I want this specific return."
- Second constraint: "My weights must sum to 100%."

> **Key Concept:** Any portfolio that does NOT lie on the efficient frontier is suboptimal. Rational investors should only hold efficient portfolios.

> **CFA Exam Tip:** The efficient frontier is the upper portion of the "bullet" shape in risk-return space. The lower portion represents dominated portfolios. If asked "which portfolio is inefficient?", look for points below and to the right of the frontier.### Formal Definition

> **Key Concept:** The **efficient frontier** is the set of all portfolios that satisfy:
> $$\min_{\mathbf{w}} \; \mathbf{w}'\Sigma\mathbf{w} \quad \text{subject to} \quad \mathbf{w}'\boldsymbol{\mu} = \mu_\text{target}, \quad \mathbf{w}'\mathbf{1} = 1$$
>
> In words: for each target return, find the portfolio with the lowest risk.

Portfolios on the efficient frontier are **Pareto optimal** — you cannot improve return without increasing risk, or reduce risk without sacrificing return. Everything below or to the right of the frontier is suboptimal.

### The Bullet Shape

The feasible set of all possible portfolios forms a "bullet" or "parabola" in $(\sigma, E(R))$ space. The efficient frontier is the upper edge of this bullet (from the MVP upward). The lower edge exists mathematically but is inefficient — same risk, lower return.


---
## 7. The Minimum Variance Portfolio (MVP)

The MVP is the portfolio with the lowest possible risk, regardless of return. It sits at the **leftmost point** of the efficient frontier.

### Why Does It Matter?

The MVP tells you the absolute minimum risk you can achieve by combining these assets. It represents the "floor" of achievable risk.

### Closed-Form Solution

$$\mathbf{w}_{\text{MVP}} = \frac{\boldsymbol{\Sigma}^{-1} \mathbf{1}}{\mathbf{1}^\top \boldsymbol{\Sigma}^{-1} \mathbf{1}}$$

**In plain English:** invert the covariance matrix, multiply by a vector of ones, then normalize so the weights sum to 1. The inverse naturally assigns higher weight to low-risk, low-correlation assets.

### What to Expect

With our 5 assets, we expect the MVP to load heavily on **Bonds** (lowest volatility at 5%, and low correlation with everything else).

> **Key Concept:** The MVP depends ONLY on the covariance matrix -- not on expected returns. Two analysts who agree on risk but disagree on returns will agree on the MVP. This makes it robust.

> **CFA Exam Tip:** The MVP is the leftmost point on the efficient frontier. It does NOT necessarily have the lowest expected return among all assets.

In [ ]:
def min_variance_portfolio(Sigma):
    """Closed-form minimum variance portfolio (unconstrained, short sales allowed).
    
    w_MVP = Sigma^{-1} @ 1 / (1^T @ Sigma^{-1} @ 1)
    """
    n = Sigma.shape[0]
    ones = np.ones(n)
    Sigma_inv = np.linalg.inv(Sigma)
    w = Sigma_inv @ ones / (ones @ Sigma_inv @ ones)
    return w

w_mvp = min_variance_portfolio(Sigma)
mu_mvp = w_mvp @ mu
sig_mvp = np.sqrt(w_mvp @ Sigma @ w_mvp)

print("Minimum Variance Portfolio:")
for name, wi in zip(asset_names, w_mvp):
    print(f"  {name:<16} {wi:>8.2%}")
print(f"\n  Expected return: {mu_mvp:.2%}")
print(f"  Volatility:      {sig_mvp:.2%}")

### Interpreting the MVP Weights

Notice how the optimizer assigns large weight to **Bonds** -- the lowest-volatility asset with the lowest correlations. The portfolio volatility is lower than ANY individual asset's volatility. That is the power of diversification.

Also note: the MVP does not care about expected returns at all. It might sacrifice significant return in pursuit of the absolute minimum risk. For most investors, a portfolio further up the efficient frontier would be preferred.This makes intuitive sense — the MVP doesn't care about returns at all; it only minimises risk. So it heavily favours the lowest-volatility asset and assets with low correlations to each other.

> **CFA Exam Tip:** The MVP is the leftmost point on the efficient frontier. All portfolios on the frontier ABOVE the MVP are "efficient" (higher return for same risk). Portfolios below the MVP are "inefficient" — they have the same risk but lower return. No rational investor would choose a portfolio below the MVP.


---
## 8. Efficient Frontier via Lagrange Multipliers

To trace the entire efficient frontier, we solve the mean-variance optimization for each target return using Lagrange multipliers.

### The KKT System

The first-order conditions give an $(N+2) \times (N+2)$ linear system:

$$\begin{bmatrix} \boldsymbol{\Sigma} & \boldsymbol{\mu} & \mathbf{1} \\\ \boldsymbol{\mu}^\top & 0 & 0 \\\ \mathbf{1}^\top & 0 & 0 \end{bmatrix} \begin{bmatrix} \mathbf{w} \\\ \lambda_1 \\\ \lambda_2 \end{bmatrix} = \begin{bmatrix} \mathbf{0} \\\ \mu_{\text{target}} \\\ 1 \end{bmatrix}$$

This has a unique closed-form solution -- just solve the linear system. No iterative optimization needed.

> **Key Concept:** The Lagrange multiplier approach gives exact, analytical solutions for the efficient frontier when short selling is allowed. The frontier is a hyperbola in mean-variance space.> **Key Concept:** The Lagrangian approach converts a constrained optimisation problem into an unconstrained one by introducing multipliers. For mean-variance: minimise $\mathbf{w}'\Sigma\mathbf{w} - \lambda_1(\mathbf{w}'\boldsymbol{\mu} - \mu_t) - \lambda_2(\mathbf{w}'\mathbf{1} - 1)$. Taking derivatives and setting to zero gives a linear system with a closed-form solution.

> **CFA Exam Tip:** You won't need to solve the Lagrangian on the exam, but you should understand that the efficient frontier is the solution to a constrained optimisation problem, and that the closed-form solution requires matrix algebra (which is why computers are essential for portfolios with more than 2-3 assets).


In [ ]:
def efficient_portfolio_analytical(mu, Sigma, target_return):
    """Solve for efficient portfolio weights via Lagrange multipliers.
    
    Builds and solves the (N+2) x (N+2) KKT system for the
    minimum-variance portfolio achieving exactly target_return.
    """
    n = len(mu)
    ones = np.ones(n)
    
    # Build KKT system
    A = np.zeros((n + 2, n + 2))
    A[:n, :n] = Sigma          # top-left: covariance matrix
    A[:n, n] = mu              # top-right column: expected returns
    A[:n, n + 1] = ones        # top-right column: ones
    A[n, :n] = mu              # bottom rows: constraint gradients
    A[n + 1, :n] = ones
    
    b = np.zeros(n + 2)
    b[n] = target_return       # target return constraint
    b[n + 1] = 1.0             # weights-sum-to-one constraint
    
    x = np.linalg.solve(A, b)
    return x[:n]  # first N entries are the optimal weights

# Trace the efficient frontier from the MVP return up to 14%
target_returns = np.linspace(mu_mvp, 0.14, 100)
ef_vols = []
ef_rets = []

for target in target_returns:
    w = efficient_portfolio_analytical(mu, Sigma, target)
    vol = np.sqrt(w @ Sigma @ w)
    ef_vols.append(vol)
    ef_rets.append(target)

ef_vols = np.array(ef_vols)
ef_rets = np.array(ef_rets)

print(f"Frontier computed: {len(ef_vols)} points")
print(f"Return range: {ef_rets[0]:.2%} to {ef_rets[-1]:.2%}")
print(f"Vol range:    {ef_vols[0]:.2%} to {ef_vols[-1]:.2%}")

---
## 9. Two-Fund Separation Theorem

One of the most elegant results in portfolio theory:

> **Any portfolio on the efficient frontier can be created as a blend of just two efficient portfolios.**

### What Does This Mean in Practice?

Suppose a fund company offers two mutual funds, both on the efficient frontier:
- **Fund A:** A conservative portfolio targeting 6% return
- **Fund B:** An aggressive portfolio targeting 12% return

Then ANY investor -- regardless of risk tolerance -- can achieve their optimal portfolio by simply choosing the right mix of Fund A and Fund B.

### Why Does It Work?

The efficient frontier weights are a *linear function* of the target return (from the linear KKT system). So:

$$\mathbf{w}(\mu_{\text{target}}) = \alpha \, \mathbf{w}_A + (1 - \alpha) \, \mathbf{w}_B$$

### A Real-World Analogy

Think of mixing paint. If you have blue (conservative) and red (aggressive), you can make any shade of purple. Two "basis" portfolios span the entire efficient frontier.

### Why This Matters

Two-fund separation justifies the "core-satellite" approach and explains why target-date retirement funds work -- they simply shift the mix between a stock fund and a bond fund over time.

> **CFA Exam Tip:** Two-fund separation is often tested conceptually. All investors hold the *same two funds*, just in different proportions. This result holds exactly only when short selling is allowed.
> **Common Mistake:** Two-fund separation applies to the efficient frontier of risky assets. Once a risk-free asset is available, we get **one-fund separation**: every investor holds the tangency portfolio combined with the risk-free asset in different proportions.


In [ ]:
# Pick two arbitrary portfolios on the efficient frontier
w_A = efficient_portfolio_analytical(mu, Sigma, 0.06)  # conservative
w_B = efficient_portfolio_analytical(mu, Sigma, 0.12)  # aggressive

# Blend them with varying alpha to verify they trace the frontier
alphas = np.linspace(-0.5, 1.5, 200)
blend_rets = []
blend_vols = []
for a in alphas:
    w = a * w_A + (1 - a) * w_B
    blend_rets.append(w @ mu)
    blend_vols.append(np.sqrt(w @ Sigma @ w))

fig, ax = plt.subplots()
ax.plot(np.array(blend_vols) * 100, np.array(blend_rets) * 100, '--', 
        color='grey', linewidth=2, label='Two-fund combinations')
ax.plot(ef_vols * 100, ef_rets * 100, color=PRIMARY, linewidth=3, label='Efficient frontier')

# Individual assets
for i in range(N):
    ax.plot(sigma[i] * 100, mu[i] * 100, 'o', markersize=10, color=SECONDARY)
    ax.annotate(asset_names[i], (sigma[i] * 100 + 0.3, mu[i] * 100 + 0.2), fontsize=9)

ax.plot(sig_mvp * 100, mu_mvp * 100, '*', color=ACCENT, markersize=15, zorder=5, label='MVP')
ax.set_xlabel('Portfolio Std Dev (%)')
ax.set_ylabel('Expected Return (%)')
ax.set_title('Efficient Frontier with Individual Assets')
ax.legend()
plt.tight_layout()
plt.show()

### Interpreting the Plot

The grey dashed line (two-fund blends) lies **exactly** on the efficient frontier (blue solid line). This confirms the two-fund separation theorem.

Also notice that **every individual asset** (orange dots) lies to the right of the frontier -- every asset is inefficient on its own. You can always do better by combining them.This is a remarkable result. It means:
- Any investor, regardless of preferences, only needs to choose between two "master" portfolios
- Asset managers can offer just two funds, and every client can create their optimal portfolio by mixing them
- The entire efficient frontier is accessible with just two building blocks

> **Key Concept:** Two-fund separation is why index funds work so well conceptually. In the CAPM world, one of the two funds is the market portfolio. Combined with the risk-free asset, this gives the **one-fund separation** theorem: every investor holds some mix of the market portfolio and the risk-free asset.


---
## 10. Short-Sale Constraints: The Real World

The analytical solution above allows negative weights (short selling). In reality, many investors cannot short sell.

### What Changes?

1. **The efficient frontier shifts to the right** -- higher risk for the same return.
2. **The frontier can have "kinks"** where constraints become active.
3. **No closed-form solution** -- we need numerical optimization (quadratic programming).
4. **Two-fund separation no longer holds exactly.**

> **Key Concept:** Constraints always make the investor worse off. The unconstrained frontier is the theoretical ideal; the constrained frontier is practical reality.

> **CFA Exam Tip:** Removing short selling shrinks the feasible set and moves the frontier to the right. The constrained frontier is always on or to the right of the unconstrained one.

In [ ]:
def efficient_portfolio_constrained(mu, Sigma, target_return):
    """Minimum variance portfolio with no-short-sale constraint (w_i >= 0).
    
    Uses SLSQP numerical optimizer since the closed-form no longer applies.
    """
    n = len(mu)
    
    def objective(w):
        return w @ Sigma @ w  # minimize portfolio variance
    
    constraints = [
        {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},           # weights sum to 1
        {'type': 'eq', 'fun': lambda w: w @ mu - target_return},  # target return
    ]
    bounds = [(0, 1) for _ in range(n)]  # no short selling: 0 <= w_i <= 1
    w0 = np.ones(n) / n  # start with equal weights
    
    result = optimize.minimize(objective, w0, method='SLSQP',
                               bounds=bounds, constraints=constraints)
    return result.x

# Trace the constrained (long-only) efficient frontier
target_rets_c = np.linspace(mu.min() + 0.001, mu.max() - 0.001, 80)
ef_vols_c = []
ef_rets_c = []

for target in target_rets_c:
    try:
        w = efficient_portfolio_constrained(mu, Sigma, target)
        vol = np.sqrt(w @ Sigma @ w)
        ef_vols_c.append(vol)
        ef_rets_c.append(target)
    except:
        pass

fig, ax = plt.subplots()
ax.plot(ef_vols * 100, ef_rets * 100, '--', color='grey', linewidth=2, label='Unconstrained')
ax.plot(np.array(ef_vols_c) * 100, np.array(ef_rets_c) * 100, color=PRIMARY, linewidth=3, label='Long-only')

for i in range(N):
    ax.plot(sigma[i] * 100, mu[i] * 100, 'o', markersize=10, color=SECONDARY)
    ax.annotate(asset_names[i], (sigma[i] * 100 + 0.3, mu[i] * 100 + 0.2), fontsize=9)

ax.set_xlabel('Portfolio Std Dev (%)')
ax.set_ylabel('Expected Return (%)')
ax.set_title('Efficient Frontier: Unconstrained vs Long-Only')
ax.legend()
plt.tight_layout()
plt.show()

### Interpreting the Comparison

The long-only frontier (blue) is to the *right* of the unconstrained frontier (grey dashed). For the same return, you must accept more risk when you cannot short sell. In practice, this cost is often modest for well-diversified portfolios.This is the "cost of constraints." By forbidding short sales, we lose some efficient portfolios, especially at the extremes. The long-only frontier can never be to the LEFT of the unconstrained frontier — constraints can only make things worse (or at best, the same).

> **Important:** In practice, most institutional investors have constraints: no short selling, sector limits, maximum position sizes, ESG exclusions. Each constraint potentially pushes the frontier rightward (higher risk for the same return). The art of portfolio management is achieving near-frontier performance despite real-world constraints.


---
## 11. Adding a Risk-Free Asset: The Capital Allocation Line

Everything so far involved only risky assets. Now let's introduce a **risk-free asset** -- an investment with zero volatility and a guaranteed return (think: Treasury bills).

### Why Does This Change Everything?

When you blend a risk-free asset with any risky portfolio, the result is a **straight line** in risk-return space (because the risk-free asset has zero variance and zero correlation with everything).

### The Math

A portfolio with weight $w_P$ in risky portfolio $P$ and $(1 - w_P)$ in the risk-free asset:

$$E[R_C] = R_f + w_P (E[R_P] - R_f)$$
$$\sigma_C = w_P \sigma_P$$

Eliminating $w_P$:

$$E[R_C] = R_f + \frac{E[R_P] - R_f}{\sigma_P} \times \sigma_C$$

A straight line with slope = **Sharpe ratio** of portfolio $P$.

### The Key Insight

Every investor wants the risky portfolio with the **highest Sharpe ratio** -- the **tangency portfolio**. The line from $R_f$ through the tangency portfolio is the **Capital Allocation Line (CAL)** and it dominates the entire efficient frontier.

### How Investors Use the CAL

- **Conservative** ($w_P < 1$): Part in T-bills, part in tangency portfolio ("lending")
- **Moderate** ($w_P = 1$): 100% in tangency portfolio
- **Aggressive** ($w_P > 1$): Borrow at $R_f$ to leverage the tangency portfolio ("borrowing")

> **Key Concept:** With a risk-free asset, the investment problem separates into two steps: (1) find the tangency portfolio (same for everyone), and (2) choose how much risk to take (personal preference). This is the **separation theorem**.

> **CFA Exam Tip:** The CAL through the tangency portfolio is the steepest possible line from $R_f$. When the tangency portfolio IS the market portfolio (CAPM assumption), the CAL becomes the **Capital Market Line (CML)**. Know the distinction.### The Capital Allocation Line (CAL)

When we add a risk-free asset with return $R_f$, any combination of the risk-free asset and a risky portfolio $P$ lies on a straight line:

$$E(R_c) = R_f + \frac{E(R_P) - R_f}{\sigma_P} \sigma_c$$

The slope of this line is the **Sharpe ratio** of portfolio $P$. The steeper the line (higher Sharpe), the better the risk-return trade-off.

> **Key Concept:** The CAL that is tangent to the efficient frontier offers the BEST risk-return trade-off achievable. Every point on this line dominates every point on the curved efficient frontier (except the tangency point where they touch).

**How investors use the CAL:**
- **Conservative investor:** Mostly risk-free asset, a little in the tangency portfolio (left end of CAL)
- **Moderate investor:** 50/50 split between risk-free and tangency portfolio
- **Aggressive investor:** 100% in the tangency portfolio (at the tangency point)
- **Leveraged investor:** Borrow at $R_f$ and invest MORE than 100% in the tangency portfolio (right of tangency point on the CAL)


---
## 12. The Tangency Portfolio and the Sharpe Ratio

The tangency portfolio maximizes the Sharpe ratio:

$$\text{Sharpe Ratio} = \frac{E[R_T] - R_f}{\sigma_T}$$

**In words:** excess return per unit of total risk.

### Closed-Form Solution

$$\mathbf{w}_T = \frac{\boldsymbol{\Sigma}^{-1} (\boldsymbol{\mu} - R_f \mathbf{1})}{\mathbf{1}^\top \boldsymbol{\Sigma}^{-1} (\boldsymbol{\mu} - R_f \mathbf{1})}$$

### What Determines the Weights?

Three things increase an asset's weight:
1. **Higher expected excess return** ($\mu_i - R_f$)
2. **Lower variance**
3. **Lower correlation with other assets**

### Tangency vs. MVP

- **MVP** = depends only on $\boldsymbol{\Sigma}$. Ignores returns.
- **Tangency** = depends on both $\boldsymbol{\Sigma}$ AND $\boldsymbol{\mu} - R_f$. More aggressive.

> **CFA Exam Tip:** The tangency portfolio changes when expected return estimates change. The MVP does not. This makes the MVP more stable in practice.

In [ ]:
def tangency_portfolio(mu, Sigma, rf):
    """Closed-form tangency portfolio (maximum Sharpe ratio).
    
    w_T = Sigma^{-1} @ (mu - rf) / (1^T @ Sigma^{-1} @ (mu - rf))
    """
    excess = mu - rf
    Sigma_inv = np.linalg.inv(Sigma)
    w = Sigma_inv @ excess
    w = w / np.sum(w)  # normalize so weights sum to 1
    return w

rf = 0.03  # 3% annual risk-free rate
w_tan = tangency_portfolio(mu, Sigma, rf)
mu_tan = w_tan @ mu
sig_tan = np.sqrt(w_tan @ Sigma @ w_tan)
sharpe = (mu_tan - rf) / sig_tan

print("Tangency Portfolio (Maximum Sharpe Ratio):")
for name, wi in zip(asset_names, w_tan):
    print(f"  {name:<16} {wi:>8.2%}")
print(f"\n  E[R]:   {mu_tan:.2%}")
print(f"  sigma:  {sig_tan:.2%}")
print(f"  Sharpe: {sharpe:.4f}")

### Interpreting the Tangency Portfolio

Compare the tangency weights with the MVP. The tangency portfolio tilts toward high-excess-return assets (like International Equity: 12% - 3% = 9% excess return) and away from Bonds (only 1% excess return). The MVP minimizes risk; the tangency maximizes risk-adjusted return.The tangency portfolio takes MORE risk than the MVP but is rewarded with disproportionately more return — it has the highest Sharpe ratio of any portfolio. This is the portfolio that every risk-return-aware investor should hold (combined with the risk-free asset in different proportions).

> **Key Concept:** The tangency portfolio is the point where the Capital Allocation Line (a straight line from $R_f$) is tangent to the efficient frontier. It maximises:
> $$\text{Sharpe Ratio} = \frac{E(R_p) - R_f}{\sigma_p}$$
> Every investor should hold this same risky portfolio, adjusting only the allocation between it and the risk-free asset based on their risk tolerance.


---
## 13. Putting It All Together

Let's visualize the complete picture: efficient frontier, individual assets, the tangency portfolio, and the Capital Allocation Line. This is the "master diagram" of mean-variance theory.

In [ ]:
# Capital Allocation Line: a straight line from Rf through the tangency portfolio
cal_vols = np.linspace(0, 0.25, 100)
cal_rets = rf + sharpe * cal_vols  # E[R] = Rf + Sharpe * sigma

fig, ax = plt.subplots()

# Efficient frontier (risky assets only)
ax.plot(ef_vols * 100, ef_rets * 100, color=PRIMARY, linewidth=2, label='Efficient frontier')

# Capital Allocation Line
ax.plot(cal_vols * 100, cal_rets * 100, '--', color=SECONDARY, linewidth=2, 
        label=f'CAL (Sharpe={sharpe:.3f})')

# Key points
ax.plot(sig_tan * 100, mu_tan * 100, '*', color=ACCENT, markersize=15, zorder=5, label='Tangency')
ax.plot(0, rf * 100, 'D', color=TERTIARY, markersize=10, zorder=5, label=f'Risk-free ({rf*100:.0f}%)')

# Individual assets
for i in range(N):
    ax.plot(sigma[i] * 100, mu[i] * 100, 'o', markersize=8, color='grey')
    ax.annotate(asset_names[i], (sigma[i] * 100 + 0.3, mu[i] * 100), fontsize=8)

ax.set_xlim(0, 28)
ax.set_ylim(0, 16)
ax.set_xlabel('Portfolio Std Dev (%)')
ax.set_ylabel('Expected Return (%)')
ax.set_title('Capital Allocation Line & Efficient Frontier')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

### Reading the Final Plot

This diagram summarizes the entire theory:

1. **Individual assets** (grey dots) are inefficient -- they lie to the right of the frontier.
2. **The efficient frontier** (blue curve) shows the best risky-only portfolios.
3. **The tangency portfolio** (gold star) maximizes the Sharpe ratio.
4. **The CAL** (dashed line) dominates the frontier -- by mixing the tangency portfolio with the risk-free asset, you can reach any point on this line, which lies ABOVE the frontier.
5. **Conservative investors** sit left of the tangency on the CAL (lending).
6. **Aggressive investors** sit right of the tangency (borrowing/leveraging).

### The Extended Separation Theorem

With a risk-free asset, all investors hold just **two funds** -- the risk-free asset and the tangency portfolio. A conservative retiree and an aggressive hedge fund manager hold the SAME risky portfolio -- they differ only in how much they put in cash vs. the risky portfolio.

This is the theoretical foundation for index investing.

> **Key Concept:** The CAL slope (Sharpe ratio) is the "price of risk" -- how much extra return the market offers per unit of risk.

> **CFA Exam Tip:** Distinguish between: (1) the CAL (any risky portfolio + risk-free), (2) the CML (specifically when the tangency = market portfolio -- this is CAPM), and (3) the SML (plots expected return vs. beta, not sigma). Three different lines.> **CFA Exam Tip:** This is the most important diagram in portfolio theory. Be able to identify:
> 1. The efficient frontier (upper portion of the bullet)
> 2. The minimum variance portfolio (leftmost point)
> 3. The tangency portfolio (where CAL touches the frontier)
> 4. The Capital Allocation Line (straight line from $R_f$ through tangency)
> 5. Individual assets (always INSIDE the frontier — diversification always helps)
>
> Any portfolio ON the CAL dominates any portfolio on the efficient frontier (except the tangency point where they meet). The CAL is the new efficient frontier once a risk-free asset is available.


---
## 14. Summary of Key Formulas

| Concept | Formula | Notes |
|:--|:--|:--|
| Portfolio return | $E[R_P] = \mathbf{w}^\top \boldsymbol{\mu}$ | Weighted average -- always |
| Portfolio variance | $\sigma_P^2 = \mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w}$ | NOT weighted average of variances |
| 2-asset variance | $w_1^2\sigma_1^2 + w_2^2\sigma_2^2 + 2w_1 w_2 \rho_{12}\sigma_1\sigma_2$ | Must memorize for the exam |
| MVP weights | $\frac{\boldsymbol{\Sigma}^{-1}\mathbf{1}}{\mathbf{1}^\top\boldsymbol{\Sigma}^{-1}\mathbf{1}}$ | Depends only on covariance |
| Tangency weights | $\frac{\boldsymbol{\Sigma}^{-1}(\boldsymbol{\mu}-R_f)}{\mathbf{1}^\top\boldsymbol{\Sigma}^{-1}(\boldsymbol{\mu}-R_f)}$ | Max Sharpe ratio portfolio |
| CAL equation | $E[R_C] = R_f + \text{SR}_T \times \sigma_C$ | Straight line from $R_f$ |
| Sharpe ratio | $\frac{E[R]-R_f}{\sigma}$ | Excess return per unit of total risk |> **The Big Picture:** Mean-variance optimisation is the theoretical foundation, but in practice it requires good estimates of expected returns, volatilities, and correlations. The output is only as good as the inputs — "garbage in, garbage out." This motivates robust approaches like shrinkage estimation, Black-Litterman, and risk parity, which are less sensitive to input estimation errors.


---
## 15. References

1. Markowitz, H. "Portfolio Selection," *Journal of Finance*, 1952.
2. Merton, R. C. "An Analytic Derivation of the Efficient Portfolio Frontier," *JFQA*, 1972.
3. Bodie, Z., Kane, A., Marcus, A. *Investments*, 12th ed., McGraw-Hill, 2021.
4. CFA Institute, *CFA Program Curriculum Level I -- Portfolio Management*.
5. Elton, E. J. et al. *Modern Portfolio Theory and Investment Analysis*, 9th ed., Wiley, 2014.### CFA Level 1 Curriculum Alignment

Key Learning Outcome Statements covered:
- LOS: Calculate and interpret portfolio expected return and standard deviation
- LOS: Describe the effect of correlation on portfolio risk
- LOS: Describe the efficient frontier and its properties
- LOS: Explain the capital allocation line and the capital market line
- LOS: Describe the two-fund separation theorem
